# Attention-Guided O+CA Training for Tiny Object Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/oca-tiny-object-detection/blob/main/OCA_Tiny_Object_Detection.ipynb)

This notebook runs the full experiment from start to finish on a free Google Colab GPU:

1. **Baseline**: train YOLOv8 on full VisDrone images
2. **Warm-up**: train a short model whose attention we can visualise
3. **Grad-CAM++ crops**: find where the warm-up model looks for each class, then cut zoomed-in crops around those regions
4. **O+CA**: train the same YOLOv8 on Original + Cropped-Attention images
5. **Evaluate**: compare baseline and O+CA on the uncropped validation set, with extra focus on tiny objects

> **Credit.** This is an unofficial reimplementation of the method described in: Swetha Manikandan, *An Attention-Guided Heatmap Clustering Framework for Tiny Object Detection Using Grad-CAM++*, M.S. thesis, Yuan Ze University, 2025 (advisor: Prof. Qazi Mazhar Ul Haq). Detector: [Ultralytics YOLO](https://github.com/ultralytics/ultralytics) (AGPL-3.0).

### How to use this notebook
1. **Runtime → Change runtime type → T4 GPU** (free tier), then **Save**.
2. In **Settings** below, paste your GitHub repository URL and pick a `RUN_MODE`.
3. Run the cells **from top to bottom** (Shift + Enter).
4. If Colab disconnects, reconnect and **run all cells again from the top**. Finished steps are skipped and unfinished training resumes from its last checkpoint, saved in your Google Drive.

| `RUN_MODE` | What it does | Approx. time on a free T4 |
|---|---|---|
| `smoke_test` | Tiny 8-image dataset, a few epochs, checks that everything works | ~5 min |
| `quick` | VisDrone, YOLOv8n, 30-epoch runs: a real but smaller experiment | ~8 h total, over 2–3 sessions |
| `full` | Thesis settings: YOLOv8l, 100 epochs | Days on a T4; use Colab Pro (A100) or your own GPU |

## 1. Settings

In [ ]:
#@title Settings { display-mode: "form" }
REPO_URL = "https://github.com/YOUR_USERNAME/oca-tiny-object-detection.git"  #@param {type:"string"}
RUN_MODE = "smoke_test"  #@param ["smoke_test", "quick", "full"]
K = 2  #@param [1, 2, 3] {type:"raw"}
USE_DRIVE = True  #@param {type:"boolean"}

PRESETS = {
    #              dataset          model           baseline warm-up O+CA  batch
    "smoke_test": dict(data="coco8.yaml",    model="yolov8n.yaml", ep_base=3,   ep_warm=3,  ep_oca=3,   batch=8),
    "quick":      dict(data="VisDrone.yaml", model="yolov8n.yaml", ep_base=30,  ep_warm=15, ep_oca=30,  batch=16),
    "full":       dict(data="VisDrone.yaml", model="yolov8l.yaml", ep_base=100, ep_warm=15, ep_oca=100, batch=16),
}
P = PRESETS[RUN_MODE]
DATA, MODEL, BATCH = P["data"], P["model"], P["batch"]

# Grad-CAM++ class-confidence threshold (thesis: 0.25). See step 5 if very few crops are produced.
TAU_CLS = 0.25

print(f"Mode: {RUN_MODE} | data: {DATA} | model: {MODEL} | epochs: "
      f"baseline {P['ep_base']}, warm-up {P['ep_warm']}, O+CA {P['ep_oca']} | k = {K}")

## 2. Setup
Checks the GPU, connects Google Drive (so your training survives disconnects), downloads the code from your GitHub repository, and installs the requirements.

In [ ]:
import os, sys, subprocess, torch

IN_COLAB = "google.colab" in sys.modules
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "0" else "none (CPU only)")
if DEVICE == "cpu" and RUN_MODE != "smoke_test":
    raise SystemExit("No GPU found. In Colab: Runtime → Change runtime type → T4 GPU, then run again.")

# Where checkpoints and results are kept
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = f"/content/drive/MyDrive/oca-tiny-object-detection/{RUN_MODE}"
else:
    SAVE_DIR = os.path.abspath(f"outputs/{RUN_MODE}")
os.makedirs(SAVE_DIR, exist_ok=True)
RUNS = f"{SAVE_DIR}/runs"
print("Saving checkpoints and results to:", SAVE_DIR)

In [ ]:
# Get the code: clone the repo in Colab, or use the current folder if the notebook is opened inside the repo
if os.path.exists("train.py"):
    REPO_DIR = os.getcwd()
else:
    REPO_DIR = "/content/oca-tiny-object-detection" if IN_COLAB else os.path.abspath("oca-tiny-object-detection")
    if not os.path.exists(REPO_DIR):
        if "YOUR_USERNAME" in REPO_URL:
            raise SystemExit("Put your own GitHub repository URL in the Settings cell first.")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)
os.chdir(REPO_DIR)
print("Working in:", REPO_DIR)

!pip install -q -r requirements.txt

## 3. Look at the data
The first time, Ultralytics downloads and converts the dataset automatically. VisDrone is about 2 GB and takes a few minutes. In a new Colab session it is downloaded again, because only checkpoints and results are kept in Drive.

In [ ]:
import random, cv2, numpy as np, matplotlib.pyplot as plt
from ultralytics.data.utils import check_det_dataset
from build_oca_dataset import list_images
from oca.crops import read_yolo_labels
from ultralytics.data.utils import img2label_paths

data_cfg = check_det_dataset(DATA)
train_imgs, val_imgs = list_images(data_cfg["train"]), list_images(data_cfg["val"])
print(f"{len(train_imgs)} training images, {len(val_imgs)} validation images, {data_cfg['nc']} classes")
print("Classes:", list(data_cfg["names"].values()))

def show_with_boxes(paths, title=""):
    fig, axes = plt.subplots(1, len(paths), figsize=(6 * len(paths), 5))
    for ax, p in zip(np.atleast_1d(axes), paths):
        img = cv2.imread(p); h, w = img.shape[:2]
        boxes = read_yolo_labels(img2label_paths([p])[0], w, h)
        for _, x1, y1, x2, y2 in boxes:
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), max(2, w // 500))
        ax.imshow(img[:, :, ::-1]); ax.set_title(f"{len(boxes)} labelled objects"); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

random.seed(1)
show_with_boxes(random.sample(train_imgs, 3), "Training images with ground-truth boxes")

## 4. Train the models

All runs use the thesis's training settings (AdamW, learning rate 1e-3, cosine decay, Mosaic/MixUp/HSV augmentation, trained **from scratch**). Only the training data differs between the baseline and O+CA.

Each cell can be re-run safely: if the run is finished it is skipped, and if it was interrupted it resumes.

### 4a. Baseline (full images only)

In [ ]:
!python train.py --data {DATA} --name baseline --model {MODEL} --epochs {P["ep_base"]} --batch {BATCH} \
    --project "{RUNS}" --device {DEVICE} --workers 2 --resume

### 4b. Warm-up model
A short training run. Its only job is to produce attention heatmaps in step 5 (the thesis uses 15 warm-up epochs).

In [ ]:
!python train.py --data {DATA} --name warmup --model {MODEL} --epochs {P["ep_warm"]} --batch {BATCH} \
    --project "{RUNS}" --device {DEVICE} --workers 2 --resume

WARMUP_WEIGHTS = f"{RUNS}/warmup/weights/last.pt"
if RUN_MODE == "smoke_test":
    # A 3-epoch model is too weak to be confident about anything, so the smoke test borrows
    # a pretrained model just to exercise the pipeline. Real runs use the warm-up model above.
    WARMUP_WEIGHTS = "yolov8n.pt"
print("Heatmaps will come from:", WARMUP_WEIGHTS)

## 5. Build the O+CA dataset with Grad-CAM++
For every training image, the warm-up model produces one Grad-CAM++ heatmap per detected class. The strongest regions that contain real objects become new, zoomed-in training images. Overlapping crops (IoU > 0.7) are removed and up to `K` crops are kept per image.

The crops are archived to Google Drive, so they don't have to be rebuilt in a new session.

In [ ]:
import json, shutil, tarfile
OCA_DIR = os.path.abspath(f"{'/content/datasets' if IN_COLAB else 'datasets'}/oca_k{K}")
ARCHIVE = f"{SAVE_DIR}/oca_k{K}.tar"

if os.path.exists(f"{OCA_DIR}/data_oca.yaml"):
    print("O+CA dataset already present.")
elif os.path.exists(ARCHIVE):
    print("Restoring O+CA dataset from Drive ...")
    with tarfile.open(ARCHIVE) as t:
        t.extractall("/", filter="data")
else:
    !python build_oca_dataset.py --data {DATA} --weights "{WARMUP_WEIGHTS}" --k {K} --tau-cls {TAU_CLS} --out "{OCA_DIR}" --vis 12
    print("Archiving to Drive ...")
    with tarfile.open(ARCHIVE, "w") as t:
        t.add(OCA_DIR, arcname=OCA_DIR.lstrip("/"))

stats = json.load(open(f"{OCA_DIR}/stats.json"))
print(f"\nOriginal images: {stats['original_images']}")
print(f"Attention crops: {stats['attention_crops']}")
print(f"O+CA training samples: {stats['oca_training_samples']}")
if stats["attention_crops"] < 0.5 * stats["original_images"]:
    print("\n⚠️ Few crops were produced. A short warm-up model is often not confident enough for the 0.25 "
          "threshold. Set TAU_CLS = 0.1 in the Settings cell, delete the oca_k folder and archive, then re-run this step. "
          "Write the value you used in your README.")

In [ ]:
# Heatmaps (red = where the model looks) with the selected crop windows in white
import glob
vis = sorted(glob.glob(f"{OCA_DIR}/visuals/*.jpg"))[:4]
if vis:
    fig, axes = plt.subplots(1, len(vis), figsize=(6 * len(vis), 5))
    for ax, p in zip(np.atleast_1d(axes), vis):
        ax.imshow(cv2.imread(p)[:, :, ::-1]); ax.axis("off")
    fig.suptitle("Grad-CAM++ attention and selected crops"); plt.tight_layout(); plt.show()

crops = sorted(glob.glob(f"{OCA_DIR}/images/crops/*.jpg"))
if crops:
    random.seed(2)
    show_with_boxes(random.sample(crops, min(4, len(crops))), "Attention crops with projected labels")

### 5b. Train on O+CA
Same model and same settings as the baseline. Only the training data has changed.

In [ ]:
!python train.py --data "{OCA_DIR}/data_oca.yaml" --name oca_k{K} --model {MODEL} --epochs {P["ep_oca"]} --batch {BATCH} \
    --project "{RUNS}" --device {DEVICE} --workers 2 --resume

## 6. Evaluate
Both models are scored on the **original, uncropped** validation images with the official COCO evaluator. The table includes the fine-grained tiny-object ranges from the thesis: objects smaller than 16×16 pixels, 16–32 pixels and 32–64 pixels.

In [ ]:
RESULTS = f"{SAVE_DIR}/results"
!python evaluate.py --data {DATA} --weights "{RUNS}/baseline/weights/best.pt" "{RUNS}/oca_k{K}/weights/best.pt" \
    --out "{RESULTS}/metrics.json"

from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/metrics.md").read()))
if RUN_MODE == "smoke_test":
    print("Smoke test: 3-epoch models trained from scratch usually detect nothing yet, so 'nan' scores are "
          "expected here. If you got this far without errors, everything works. Switch RUN_MODE to 'quick'.")

In [ ]:
# Baseline vs O+CA, by metric and by class
metrics = json.load(open(f"{RESULTS}/metrics.json"))
names = list(metrics)                       # ["baseline", "oca_kK"]
labels = {"baseline": "Baseline (full images)", f"oca_k{K}": f"O+CA (k={K})"}
colors = ["#2a78d6", "#eb6834"]             # fixed order: baseline = blue, O+CA = orange

def grouped_bars(ax, keys, get, title):
    x = np.arange(len(keys)); w = 0.38
    for i, n in enumerate(names):
        vals = [get(n, k) for k in keys]
        bars = ax.bar(x + (i - 0.5) * w, vals, w - 0.02, color=colors[i], label=labels.get(n, n))
    ax.set_xticks(x, keys, rotation=30, ha="right"); ax.set_title(title, loc="left")
    ax.set_ylim(0, None); ax.grid(axis="y", alpha=0.25); ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 1.3]})
grouped_bars(a1, ["AP", "AP50", "AP_S", "AP<16", "AP16-32", "AP32-64"],
             lambda n, k: metrics[n].get(k, np.nan), "Overall and tiny-object AP")
classes = [c for c in metrics[names[0]].get("per_class_AP", {})          # skip classes absent from val set
           if any(not np.isnan(metrics[n]["per_class_AP"].get(c, np.nan)) for n in names)]
grouped_bars(a2, classes, lambda n, k: metrics[n]["per_class_AP"].get(k, np.nan), "AP per class (IoU 0.50:0.95)")
a1.set_ylabel("Average precision"); a1.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"{RESULTS}/comparison.png", dpi=150, bbox_inches="tight"); plt.show()
print("Saved", f"{RESULTS}/comparison.png")

## 7. Add your results to GitHub
Your results are in Google Drive under `MyDrive/oca-tiny-object-detection/<RUN_MODE>/results/`.

1. Download `metrics.md` and `comparison.png` from Drive.
2. In your GitHub repo, upload `comparison.png` and one heatmap image (from `visuals/`) into an `images/` folder.
3. Edit `README.md` on GitHub: paste the table from `metrics.md` into the **Results** section and un-comment the image line.
4. Write down which `RUN_MODE` you used. A `quick` run is a smaller experiment than the thesis, so say so.
5. Optional: **File → Save a copy in GitHub** saves this notebook *with its outputs* to your repo, so visitors can see the plots without running anything.

**Never upload** model weights (`.pt`) or the dataset to GitHub. They are too large, and the dataset has its own license.

## Troubleshooting
| Problem | Fix |
|---|---|
| `No GPU found` | Runtime → Change runtime type → T4 GPU |
| Colab disconnected | Reconnect and run all cells from the top. Training resumes from Drive. |
| `CUDA out of memory` | Lower the batch size in `PRESETS` (e.g. 8) |
| Very few crops in step 5 | Set `TAU_CLS = 0.1` (see the message in step 5) |
| `git clone` fails | Check `REPO_URL`, and make sure the repository is **public** |
| Want to start a run over | Delete its folder in `MyDrive/oca-tiny-object-detection/<RUN_MODE>/runs/` |